In [ ]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())

GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "gold_alerts")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Query_Gold")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
gold_df = spark.read.format("delta").load(GOLD_PATH)

# Sort by the most heavy/aggressive threat vectors
top_attackers = gold_df.orderBy(F.col("request_count").desc())

top_attackers.select(
    "alert_window_start",
    "alert_window_end",
    "attacker_ip",
    "honeypot_region",
    "attacker_country",
    "request_count"
).show(10, truncate=False)

26/10/21 00:29:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 960485 ms exceeds timeout 120000 ms
26/10/21 00:29:36 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/21 00:29:37 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o